# 동적 웹페이지 크롤링  : (2) AJAX (에이젝스)
- https://webscraper.io/test-sites/e-commerce/ajax/computers/laptops
  

# 라이브러리 불러오기 

In [1]:
from datetime import datetime
from pathlib import Path

import pandas as pd

from selenium import webdriver
from selenium.common.exceptions import StaleElementReferenceException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# 기본 설정

In [49]:
TARGET_URL = 'https://webscraper.io/test-sites/e-commerce/ajax/computers/laptops'

## 동적 요소 최대 대기 시간(초)
WAIT_TIMEOUT = 10

## 브라우저 화면 표시 여부
## False : 브라우저 화면 표시
## True  : 브라우저 화면에 표시하지 않고 실행
HEADLESS = True

## csv 저장 폴더
PROJECT_DIR = Path.cwd().resolve().parents[1]
OUTPUT_DIR = PROJECT_DIR / 'data' / 'dynamic'

# Chrome WebDriver 생성

In [50]:
options = Options()

if HEADLESS:
    options.add_argument('--headless=new')

options.add_argument('--start-maximized')

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, WAIT_TIMEOUT)

print('Chrome WebDriver 생성 완료!🍀')

Chrome WebDriver 생성 완료!🍀


# AJAX 페이지 접속

In [51]:
driver.get(TARGET_URL)

In [52]:
PRODUCT_SELECTOR = 'div.product-wrapper'
PAGE_BUTTON_SELECTOR = 'button.page'

# 상품 한 건 추출

In [53]:
first_product = driver.find_element(By.CSS_SELECTOR, PRODUCT_SELECTOR)
first_product

<selenium.webdriver.remote.webelement.WebElement (session="8746318febfd2771de71a96648333874", element="f.ED27BB0BCE4A8447BFBB8492628BED59.d.16F7099A579A5C5C909CB425065CB072.e.12")>

## 상품명

In [54]:
title_element = first_product.find_element(By.CSS_SELECTOR, 'a.title')
title_element.text  ## Asus VivoBook X4
title = title_element.get_attribute('title')
title

'Asus VivoBook X441NA-GA190'

## 가격

In [55]:
price_text = first_product.find_element(By.CSS_SELECTOR, 'h4.price').text
price_text

'$295.99'

## 상품 상세

In [56]:
description = first_product.find_element(By.CSS_SELECTOR, 'p.description').text
description

'Asus VivoBook X441NA-GA190 Chocolate Black, 14", Celeron N3450, 4GB, 128GB SSD, Endless OS, ENG kbd'

## 상품 URL

In [57]:
detail_url = title_element.get_attribute('href')
detail_url

'https://webscraper.io/test-sites/e-commerce/ajax/product/60'

## 리뷰

In [58]:
review_text = first_product.find_element(By.CSS_SELECTOR, 'div.ratings span').text
review_text

'1'

In [59]:
PRODUCT_SELECTOR

'div.product-wrapper'

# 1페이지 상품 정보 추출

In [60]:
# page_1_product_elements = driver.find_elements(By.CSS_SELECTOR, PRODUCT_SELECTOR)

In [61]:
page_1_product_elements = wait.until(
    EC.presence_of_all_elements_located([By.CSS_SELECTOR, PRODUCT_SELECTOR])
)

print(f'1페이지 상품 수 : {len(page_1_product_elements)}')

1페이지 상품 수 : 6


In [62]:
page_1_products = []

for product_element in page_1_product_elements:
    # print(product_element.find_element(By.CSS_SELECTOR, 'h4.price').text)
    title_element = product_element.find_element(By.CSS_SELECTOR, 'a.title')
    title = title_element.get_attribute('title')
    price_text = product_element.find_element(By.CSS_SELECTOR, 'h4.price').text
    description = product_element.find_element(By.CSS_SELECTOR, 'p.description').text
    detail_url = title_element.get_attribute('href')
    review_text = product_element.find_element(By.CSS_SELECTOR, 'div.ratings span').text

    page_1_products.append({
        'title': title,
        'price_text': price_text,
        'description': description,
        'detail_url': detail_url,
        'review': review_text,
        'page': 1,
        'source_url': driver.current_url,
    })
        
print(f'1페이지 수집 완료 : {len(page_1_products)} 건')    

1페이지 수집 완료 : 6 건


In [63]:
page_1_products

[{'title': 'Asus VivoBook X441NA-GA190',
  'price_text': '$295.99',
  'description': 'Asus VivoBook X441NA-GA190 Chocolate Black, 14", Celeron N3450, 4GB, 128GB SSD, Endless OS, ENG kbd',
  'detail_url': 'https://webscraper.io/test-sites/e-commerce/ajax/product/60',
  'review': '1',
  'page': 1,
  'source_url': 'https://webscraper.io/test-sites/e-commerce/ajax/computers/laptops'},
 {'title': 'Prestigio SmartBook 133S Dark Grey',
  'price_text': '$299',
  'description': 'Prestigio SmartBook 133S Dark Grey, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam',
  'detail_url': 'https://webscraper.io/test-sites/e-commerce/ajax/product/61',
  'review': '9',
  'page': 1,
  'source_url': 'https://webscraper.io/test-sites/e-commerce/ajax/computers/laptops'},
 {'title': 'Prestigio SmartBook 133S Gold',
  'price_text': '$299',
  'description': 'Prestigio SmartBook 133S Gold, 13.3" FHD IPS, Celeron N3350 1.1GHz, 4GB, 32GB, Windows 10 Pro + Office 365 1 gadam',
  'd

# 페이지 버튼 확인

In [64]:
PAGE_BUTTON_SELECTOR

'button.page'

In [65]:
## 모든 페이지 버튼 엘리먼트 추출
page_buttons = driver.find_elements(By.CSS_SELECTOR, PAGE_BUTTON_SELECTOR)
len(page_buttons)

20

## 모든 페이지 버튼 텍스트 추출

In [66]:
button_texts = []

for button in page_buttons:
    # print(f'텍스트 공백 확인 : |{button.text}|')
    button_text = button.text.strip()

    if button_text:
        button_texts.append(button_text)

print(f'페이지 버튼 : {button_texts}')

페이지 버튼 : ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20']


## 다음으로 이동할 2페이지 버튼 찾기

In [67]:
page_2_button = None

for button in page_buttons:
    # print(button.text)
    if button.text.strip() == '2':
        page_2_button = button
        break

if page_2_button is None:
    raise ValueError('2페이지 버튼을 찾지 못했습니다.')

print(f'찾은 버튼 : {page_2_button.text}')

찾은 버튼 : 2


# 2페이지 이동 전 상태 저장

AJAX 페이지가 변경되면 기존 상품 DOM이 제거되는지 확인하기 위해  
`1페이지의 첫 번째 상품 요소`와 `현재 URL`을 저장한다.

In [68]:
## 현재 url
before_url = driver.current_url

In [69]:
## 1페이지 첫 번째 상품 요소
old_first_product = page_1_product_elements[0]

## 2페이지 버튼 클릭

2페이지 버튼을 클릭하면 전체 브라우저 문서를 새로 불러오는 대신  
JavaScript가 AJAX 요청을 보내고 상품 영역을 새로운 DOM으로 교체한다.

In [70]:
page_2_button.click()

## 1페이지 첫 번째 상품 DOM이 제거될 때까지 대기

In [71]:
wait.until(EC.staleness_of(old_first_product))

True

## 페이지 이동 전후 url 비교

In [72]:
after_url = driver.current_url

print(f'이동 전 URL : {before_url}')
print(f'이동 후 URL : {after_url}')
print(f'URL 변경 여부 : {before_url != after_url}')

이동 전 URL : https://webscraper.io/test-sites/e-commerce/ajax/computers/laptops
이동 후 URL : https://webscraper.io/test-sites/e-commerce/ajax/computers/laptops
URL 변경 여부 : False


In [73]:
page_1_product_elements[0]

<selenium.webdriver.remote.webelement.WebElement (session="8746318febfd2771de71a96648333874", element="f.ED27BB0BCE4A8447BFBB8492628BED59.d.16F7099A579A5C5C909CB425065CB072.e.12")>

In [74]:
## [Error] 기존 엘리먼트 접근!!! 
# page_1_product_elements[0].find_element(By.CSS_SELECTOR, 'h4.price')

# 2페이지 상품 DOM 다시 조회

기존 DOM이 제거된 것을 확인했으므로,  
이제 새로 생성된 2페이지 상품 요소를 다시 찾는다.

In [75]:
page_2_product_elements = wait.until(
    EC.presence_of_all_elements_located([By.CSS_SELECTOR, PRODUCT_SELECTOR])
)

print(f'2페이지 상품 요소 수 : {(len(page_2_product_elements))}')

2페이지 상품 요소 수 : 6


In [76]:
page_2_products = []

for product_element in page_2_product_elements:
    title_element = product_element.find_element(By.CSS_SELECTOR, 'a.title')
    title = title_element.get_attribute('title')
    price_text = product_element.find_element(By.CSS_SELECTOR, 'h4.price').text
    description = product_element.find_element(By.CSS_SELECTOR, 'p.description').text
    detail_url = title_element.get_attribute('href')
    review_text = product_element.find_element(By.CSS_SELECTOR, 'div.ratings span').text

    page_2_products.append({
        'title': title,
        'price_text': price_text,
        'description': description,
        'detail_url': detail_url,
        'review': review_text,
        'page': 2,
        'source_url': driver.current_url,
    })
        
print(f'2페이지 수집 완료 : {len(page_2_products)} 건')   

2페이지 수집 완료 : 6 건


In [77]:
print(f'1페이지 첫 상품 : {page_1_products[0]['title']}')
print(f'2페이지 첫 상품 : {page_2_products[0]['title']}')

1페이지 첫 상품 : Asus VivoBook X441NA-GA190
2페이지 첫 상품 : Hewlett Packard 250 G6 Dark Ash Silver


# 1~3 페이지 상품 수집

## 1페이지로 이동

In [78]:
driver.get(TARGET_URL)

wait.until(EC.presence_of_element_located([By.CSS_SELECTOR, PRODUCT_SELECTOR]))
print('1페이지로 다시 이동했습니다.')

1페이지로 다시 이동했습니다.


## 1~3페이지 반복 수집

In [79]:
products = []

MAX_PAGES = 3

for page in range(1, MAX_PAGES + 1):
    ## 현재 페이지의 새 상품 DOM 조회
    product_elements = wait.until(
        EC.presence_of_all_elements_located([By.CSS_SELECTOR, PRODUCT_SELECTOR])
    )

    source_url = driver.current_url
    page_products = []

    for product_element in product_elements:
        title_element = product_element.find_element(By.CSS_SELECTOR, 'a.title')
        title = title_element.get_attribute('title')
        price_text = product_element.find_element(By.CSS_SELECTOR, 'h4.price').text
        description = product_element.find_element(By.CSS_SELECTOR, 'p.description').text
        detail_url = title_element.get_attribute('href')
        review_text = product_element.find_element(By.CSS_SELECTOR, 'div.ratings span').text
    
        page_products.append({
            'title': title,
            'price_text': price_text,
            'description': description,
            'detail_url': detail_url,
            'review': review_text,
            'page': page,
            'source_url': driver.current_url,
        })

    products.extend(page_products)
    
    print(f'{page}페이지 수집 완료 : {len(page_products)}건')

    ## 마지막 페이지이면 페이지 이동 없이 반복 종료
    if page >= MAX_PAGES:
        break

    next_page = page + 1
    next_button = None

    ## 다음 페이지 번호 버튼 찾기
    page_buttons = driver.find_elements(By.CSS_SELECTOR, PAGE_BUTTON_SELECTOR)

    for button in page_buttons:
        if button.text.strip() == str(next_page):
            next_button = button
            break

    if next_button is None:
        print(f'{next_button}페이지 버튼이 없어 수집을 종료합니다.')
        break

    ## AJAX 변경 전 첫 번째 상품 DOM 저장
    old_first_product = product_elements[0]

    ## 다음 페이지 버튼 클릭
    next_button.click()

    ## 기존 상품 DOM 제거 대기
    wait.until(EC.staleness_of(old_first_product))

    ## 새 상품 DOM 생성 대기
    wait.until(EC.presence_of_element_located([By.CSS_SELECTOR, PRODUCT_SELECTOR]))
    
print()
print(f'전체 수집 상품 수 : {len(products)}')

1페이지 수집 완료 : 6건
2페이지 수집 완료 : 6건
3페이지 수집 완료 : 6건

전체 수집 상품 수 : 18


## DataFrame 생성

In [80]:
products_df = pd.DataFrame(products)
products_df.shape

(18, 7)

In [81]:
products_df[:2]

,title,price_text,description,detail_url,review,page,source_url
0,Asus VivoBook X441NA-GA190,$295.99,"Asus VivoBook X441NA-GA190 Chocolate Black, 14...",https://webscraper.io/test-sites/e-commerce/aj...,1,1,https://webscraper.io/test-sites/e-commerce/aj...
1,Prestigio SmartBook 133S Dark Grey,$299,"Prestigio SmartBook 133S Dark Grey, 13.3"" FHD ...",https://webscraper.io/test-sites/e-commerce/aj...,9,1,https://webscraper.io/test-sites/e-commerce/aj...


In [82]:
print(f'DataFrame 행 수 : {len(products_df)}')
print()
print('페이지별 수집 건수')
print(products_df['page'].value_counts())

DataFrame 행 수 : 18

페이지별 수집 건수
page
1    6
2    6
3    6
Name: count, dtype: int64


## 간단한 데이터 정리
- `$` 제거 후 가격을 숫자형으로 변환
- 리뷰 수 타입을 숫자형으로 변환

### price_text 컬럼 : $ 제거 후 숫자형 변환

In [83]:
products_df.price_text[:4]

0    $295.99
1       $299
2       $299
3    $306.99
Name: price_text, dtype: str

In [84]:
products_df.columns

Index(['title', 'price_text', 'description', 'detail_url', 'review', 'page',
       'source_url'],
      dtype='str')

In [85]:
products_df['price'] = (
    products_df['price_text']
    .str
    .replace('$', '')
    .pipe(pd.to_numeric, errors='coerce')
    .astype('Float64')
)

In [86]:
products_df.columns

Index(['title', 'price_text', 'description', 'detail_url', 'review', 'page',
       'source_url', 'price'],
      dtype='str')

### review 컬럼 : 정수형으로 변환

In [87]:
products_df['review'] = products_df.review.astype('Int64')
products_df.review.dtype

Int64Dtype()

## csv 파일 저장

In [88]:
PROJECT_DIR = Path.cwd().parents[1]
OUTPUT_DIR = PROJECT_DIR / 'data' / 'dynamic'
OUTPUT_DIR

WindowsPath('D:/AI/data_analytics/crawling/01-data-collection-pipeline/data/dynamic')

In [89]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
saved_at = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = OUTPUT_DIR / f'webscraper_ajax_laptops_{saved_at}.csv'

products_df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f'csv 저장 완료 : {output_file}')

csv 저장 완료 : D:\AI\data_analytics\crawling\01-data-collection-pipeline\data\dynamic\webscraper_ajax_laptops_20260812_092946.csv


## 저장 결과 다시 읽어 확인

In [90]:
saved_products_df = pd.read_csv(output_file)
saved_products_df.shape

(18, 8)

In [91]:
saved_products_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        18 non-null     str    
 1   price_text   18 non-null     str    
 2   description  18 non-null     str    
 3   detail_url   18 non-null     str    
 4   review       18 non-null     int64  
 5   page         18 non-null     int64  
 6   source_url   18 non-null     str    
 7   price        18 non-null     float64
dtypes: float64(1), int64(2), str(5)
memory usage: 1.3 KB


In [92]:
if(len(saved_products_df)) != len(products_df):
    raise ValueError('csv 저장 전후의 행 수가 다릅니다.')

print('csv 저장/재읽기 검증 완료')
print(f'저장 행 수 : {len(saved_products_df)}')

csv 저장/재읽기 검증 완료
저장 행 수 : 18


In [93]:
driver.quit()
print('브라우저 종료')


브라우저 종료


In [94]:
PAGE_BUTTON_SELECTOR

'button.page'

In [95]:
driver.quit()
print('종료!!!')

종료!!!
